In [ ]:
% pip install "transformers>=4.45.0"
% pip install flash-attn==2.6.3 --no-build-isolation
% pip install -U datasets polars
% pip install -U pydantic


In [ ]:
import requests
from PIL import Image
from io import BytesIO
import torch
from transformers import AutoModel
from transformers.image_utils import load_image

import time
import os
import argparse
import torch
import fitz  # pymupdf

from dotenv import load_dotenv, find_dotenv
from pathlib import Path
from PIL import Image

# Loading API-Keys and Tokens via local .env
load_dotenv(find_dotenv())

#### Helping Functions ##########################################

# Some segmentation for log readablility
def banner(title):
    print()
    print("=" * 60)
    print(f"  {title}")
    print("=" * 60)

# ── Arguments for Dev'ing
parser = argparse.ArgumentParser()
parser.add_argument("-t", action="store_true", help="Toggle Testing Path")
args = parser.parse_args()

#### 0. GLOBAL VARIABLES ########################################

MODEL_NAME = 'nvidia/llama-nemotron-colembed-vl-3b-v2'
ATTN_IMPL  = "flash_attention_2"

model = AutoModel.from_pretrained(
    MODEL_NAME,
    device_map='cuda:0',
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    attn_implementation=ATTN_IMPL
).eval()

# Queries
queries = [
    'How is AI improving the intelligence and capabilities of robots?',
    'Canary, a multilingual model that transcribes speech in English, Spanish, German, and French with punctuation and capitalization.',
    'Generative AI can generate DNA sequences that can be translated into proteins for bioengineering.'
]

image_urls = [
    "https://developer.download.nvidia.com/images/isaac/nvidia-isaac-lab-1920x1080.jpg",
    "https://developer-blogs.nvidia.com/wp-content/uploads/2024/03/asr-nemo-canary-featured.jpg",
    "https://blogs.nvidia.com/wp-content/uploads/2023/02/genome-sequencing-helix.jpg"
]

# Load all images (load_image handles both local paths and URLs)
images = [load_image(img_path) for img_path in image_urls]

# Encoding
query_embeddings = model.forward_queries(queries, batch_size=8)
image_embeddings = model.forward_images(images, batch_size=8)